## Knowledge Distillation

Knowledge Distillation(KD, 지식 증류)는 고성능의 모델(Teacher)에서 지식을 전달 받아 상대적으로 간단한 모델(Student)을 학습시키는 방법

Teacher 모델이 내부 구조, 파라미터를 공개 여부에 따라 White-box, Black-box, Gray-box로 구분됩니다. Black-box는 결과만 확인 가능한 경우며  <br>
White-box는 모델의 내부 구조나 파라미터를 전부 알 수 있는 경우입니다. Gray-box는 그 중간으로 일부만 공개되어 있는 경우입니다. <br>
이러한 Teacher 모델의 특징에 의해 KD에 활용할 수 있는 정보의 종류가 달라지게 됩니다.


#### Reference:
https://docs.pytorch.org/tutorials/beginner/knowledge_distillation_tutorial.html <br>
https://github.com/NoCodeProgram/deepLearning/blob/main/transformer/KD_toy.ipynb

Packages import

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.data import DataLoader
from torch import Tensor

Torch Device

In [6]:
if torch.backends.mps.is_available():
  my_device = torch.device('mps')
elif torch.cuda.is_available():
  my_device = torch.device('cuda')
else:
  my_device = torch.device('cpu')

3print(f"Using device: {my_device}")

Using device: mps


#### CIFAR-10 dataset

- 10개의 클래스
- 32x32 픽셀 이미지
- 10,000개의 이미지
- 50,000개의 트레이닝 이미지
- 10,000개의 테스트 이미지

입력 이미지는 RGB이므로 3개의 채널과 32x32 픽셀입니다. 기본적으로 각 이미지는 0에서 255까지의 3 x 32 x 32 = 3072개의 숫자로 표현됩니다. <br>
신경망에서 일반적인 관행은 입력을 정규화하는 것인데, 일반적으로 사용되는 활성화 함수에서 포화를 피하고 수치적 안정성을 높이는 것을 포함한 여러 가지 이유로 수행됩니다. <br>
정규화 프로세스는 각 채널의 평균을 빼고 표준 편차로 나누는 것으로 구성됩니다. 텐서 "mean=[0.485, 0.456, 0.406]"과 "std=[0.229, 0.224, 0.225]"는 이미 계산되었으며, <br>
이는 훈련 세트로 의도된 CIFAR-10의 사전 정의된 하위 세트에서 각 채널의 평균과 표준 편차를 나타냅니다. 평균과 표준 편차를 처음부터 다시 계산하지 않고 테스트 세트에도 이러한 값을 사용하는 방법에 주목하십시오. 이는 네트워크가 위의 숫자를 뺀 후 나누어 생성된 특징을 기반으로 학습되었기 때문이며, 일관성을 유지하고자 하기 때문입니다. 또한, 실제로는 테스트 세트의 평균과 표준 편차를 계산할 수 없습니다. 왜냐하면 우리의 가정에 따르면 해당 시점에는 테스트 세트에 접근할 수 없기 때문입니다.

  ![CIFAR-10](https://github.com/ultralytics/docs/releases/download/0/cifar10-sample-image.avif)

  Data References:
    1. https://www.cs.toronto.edu/~kriz/cifar.html <br>
    2. https://developer-together.tistory.com/49 <br>
    3. https://tutorials.pytorch.kr/beginner/blitz/cifar10_tutorial.html?highlight=cifar

Training & testing data loading

In [7]:
import torch.utils


# batch size
batch_size = 256

# dataset for training
transform_train = transforms.Compose([
  transforms.RandomVerticalFlip(),
  transforms.RandomHorizontalFlip(),
  transforms.ToTensor(),
  transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
train_loader = torch.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=True, num_workers=2)

# dataset for validation
transform_test = transforms.Compose([
  transforms.ToTensor(),
  transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)
test_loader = torch.utils.data.DataLoader(testset, batch_size=batch_size, shuffle=False, num_workers=2)


  0%|          | 0.00/170M [00:00<?, ?B/s]0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
100%|██████████| 170M/170M [00:53<00:00, 3.21MB/s] 
